[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tol/osint/blob/main/notebooks/pdf_table_to_excel.ipynb)

# PDF table to Excel

Upload a PDF, extract tables from all pages, and export the combined result to Excel or CSV. The notebook uses Camelot and pdfplumber first, and includes an OCR fallback for scanned PDFs.

In [ ]:
%pip -q install pandas openpyxl pdfplumber camelot-py tabula-py pypdf pillow ipywidgets pdf2image pytesseract
import os, re, subprocess, warnings
from pathlib import Path
import pandas as pd
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from IPython.display import display, FileLink
import ipywidgets as widgets
warnings.filterwarnings('ignore')
OUT = Path('notebook_output')
OUT.mkdir(exist_ok=True)
print('Ready')

## Upload PDF

In [ ]:
uploader = widgets.FileUpload(accept='.pdf', multiple=False, description='Upload PDF')
display(uploader)

In [ ]:
def save_uploaded_pdf(uploader_widget, out_dir=OUT):
    if not uploader_widget.value:
        raise ValueError('Please upload a PDF first.')
    item = list(uploader_widget.value.values())[0]
    name = item['metadata']['name'] if 'metadata' in item and 'name' in item['metadata'] else item.get('name', 'input.pdf')
    path = out_dir / name
    path.write_bytes(item['content'])
    return path

pdf_path = save_uploaded_pdf(uploader)
print(f'Saved to: {pdf_path}')

## Extraction helpers

In [ ]:
def clean_df(df):
    df = df.copy()
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    df.columns = [str(c).strip().replace('\n', ' ') for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.replace('\n', ' ', regex=False).str.strip()
        df[c] = df[c].replace({'nan': ''})
    return df

def score_table(df):
    if df is None or df.empty:
        return -1
    rows, cols = df.shape
    filled = (df.astype(str).replace('', pd.NA).notna()).sum().sum()
    return rows * max(cols, 1) + filled

def normalize_headers(df):
    if df.empty:
        return df
    cols = [re.sub(r'\s+', ' ', str(c)).strip() for c in df.columns]
    df.columns = cols
    return df


In [ ]:
def extract_with_pdfplumber(pdf_path):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            page_tables = page.extract_tables() or []
            for t in page_tables:
                if not t or len(t) < 2:
                    continue
                df = pd.DataFrame(t[1:], columns=t[0])
                df = normalize_headers(clean_df(df))
                if not df.empty:
                    df.insert(0, 'source_page', page_num)
                    tables.append(df)
    return tables

def extract_with_camelot(pdf_path):
    import camelot
    tables = []
    for flavor in ['lattice', 'stream']:
        try:
            found = camelot.read_pdf(str(pdf_path), pages='all', flavor=flavor)
            for i, t in enumerate(found):
                df = normalize_headers(clean_df(t.df))
                if df.empty or df.shape[0] < 2:
                    continue
                header = df.iloc[0].tolist()
                body = df.iloc[1:].copy()
                body.columns = header
                body = normalize_headers(clean_df(body))
                m = re.search(r'page-(\d+)', getattr(t, 'parsing_report', {}).get('page', '') if isinstance(getattr(t, 'parsing_report', {}), dict) else '')
                page_num = getattr(t, 'page', '') or ''
                try:
                    page_num = int(str(page_num).split(',')[0])
                except:
                    page_num = None
                body.insert(0, 'source_page', page_num)
                tables.append(body)
            if tables:
                break
        except Exception:
            pass
    return tables


## OCR fallback helpers

In [ ]:
def ocr_page_to_rows(image, page_num):
    text = pytesseract.image_to_string(image, config='--psm 6')
    lines = [re.sub(r'\s+', ' ', x).strip() for x in text.splitlines()]
    lines = [x for x in lines if x]
    rows = []
    for line in lines:
        parts = re.split(r'\s{2,}|	', line)
        if len(parts) >= 3:
            rows.append(parts)
    if not rows:
        return pd.DataFrame()
    width = max(len(r) for r in rows)
    rows = [r + [''] * (width - len(r)) for r in rows]
    df = pd.DataFrame(rows)
    df.insert(0, 'source_page', page_num)
    return clean_df(df)

def extract_with_ocr(pdf_path, dpi=200):
    tables = []
    images = convert_from_path(str(pdf_path), dpi=dpi)
    for page_num, image in enumerate(images, start=1):
        df = ocr_page_to_rows(image, page_num)
        if not df.empty:
            tables.append(df)
    return tables


In [ ]:
def extract_tables(pdf_path):
    tables = extract_with_camelot(pdf_path)
    if not tables:
        tables = extract_with_pdfplumber(pdf_path)
    if not tables:
        tables = extract_with_ocr(pdf_path)
    return combine_similar_tables(tables)


## Run extraction

In [ ]:
df = extract_tables(pdf_path)
if df.empty:
    raise ValueError('No tables were extracted. Try a vector PDF, install Java/Ghostscript for more formats, or adapt the extraction settings.')
display(df.head())
print(df.shape)

## Save to Excel and CSV

In [ ]:
stem = Path(pdf_path).stem
xlsx_path = OUT / f'{stem}_tables.xlsx'
csv_path = OUT / f'{stem}_tables.csv'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='tables')
df.to_csv(csv_path, index=False)
print(xlsx_path)
print(csv_path)
display(FileLink(str(xlsx_path)))
display(FileLink(str(csv_path)))

## Notes
- Best results come from PDFs with clear tabular structure.
- The notebook first tries Camelot and pdfplumber, then falls back to OCR with Tesseract for scanned PDFs.
- OCR output may need extra cleanup for complex layouts or merged cells.
- If your PDF contains multiple different tables, you can keep all extracted tables instead of combining only the dominant structure by editing `combine_similar_tables()`.